# Experimentos de detecção de veículos

Este notebook registra ground truth, benchmarks e visualizações. A aplicação final fica em `src/`.

A imagem usada é o JPEG extraído do PDF da prova (2048×1534 px). Não registre conclusões ou métricas sem executar as células correspondentes.

In [ ]:
%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.widgets import RectangleSelector
from PIL import Image

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.ground_truth import save_ground_truth  # noqa: E402

IMAGE_PATH = ROOT / "data/raw/drone_scene.jpg"
GROUND_TRUTH_PATH = ROOT / "data/annotations/ground_truth.json"
ANNOTATION_METHOD = "model_assisted_preannotations_manually_reviewed"
image = Image.open(IMAGE_PATH).convert("RGB")
image_width, image_height = image.size
print(f"Image: {image_width}x{image_height}")

## Anotação manual

Esta etapa é opcional: o ground truth revisado já está salvo no projeto. Para editar as caixas, execute `annotator = launch_manual_annotation()` em uma célula; o editor usa um widget interativo. Use `d` para apagar a última caixa e `s` para salvar. Antes de salvar, revise toda a rua e o estacionamento; cada veículo deve aparecer exatamente uma vez.

In [ ]:
class ManualVehicleAnnotator:
    """Interactive rectangle annotator that persists vehicle-only ground truth."""

    def __init__(self, image: Image.Image, output_path: Path) -> None:
        self.image = image
        self.output_path = output_path
        self.boxes = self._load_existing()
        self.figure, self.axis = plt.subplots(figsize=(16, 12))
        self.axis.imshow(image)
        self.axis.set_title("Drag: add vehicle | d: delete last | s: save")
        self.axis.set_axis_off()
        self.selector = RectangleSelector(
            self.axis, self._on_select, useblit=True, button=[1], interactive=False
        )
        self.figure.canvas.mpl_connect("key_press_event", self._on_key)
        self._redraw()

    def _load_existing(self) -> list[list[float]]:
        if not self.output_path.exists():
            return []
        payload = json.loads(self.output_path.read_text(encoding="utf-8"))
        return [annotation["bbox_xyxy"] for annotation in payload["annotations"]]

    def _on_select(self, start, end) -> None:
        if None in (start.xdata, start.ydata, end.xdata, end.ydata):
            return
        x1, x2 = sorted((start.xdata, end.xdata))
        y1, y2 = sorted((start.ydata, end.ydata))
        if x2 - x1 >= 3 and y2 - y1 >= 3:
            self.boxes.append([round(x1, 2), round(y1, 2), round(x2, 2), round(y2, 2)])
            self._redraw()

    def _on_key(self, event) -> None:
        if event.key == "d" and self.boxes:
            self.boxes.pop()
            self._redraw()
        elif event.key == "s":
            self.save()

    def _redraw(self) -> None:
        for patch in list(self.axis.patches):
            patch.remove()
        for index, (x1, y1, x2, y2) in enumerate(self.boxes, start=1):
            self.axis.add_patch(
                Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec="#00e5ff", lw=1.5)
            )
            self.axis.text(
                x1, y1, str(index), color="white", fontsize=8, bbox={"facecolor": "#004d5c"}
            )
        self.axis.set_title(f"{len(self.boxes)} vehicles | Drag: add | d: delete | s: save")
        self.figure.canvas.draw_idle()

    def save(self) -> None:
        save_ground_truth(
            self.output_path,
            IMAGE_PATH,
            self.boxes,
            annotation_method=ANNOTATION_METHOD,
            relative_to=ROOT,
        )
        print(f"Saved {len(self.boxes)} annotations to {self.output_path.relative_to(ROOT)}")


def launch_manual_annotation() -> ManualVehicleAnnotator:
    """Open the optional widget-based editor without affecting benchmark rendering."""
    get_ipython().run_line_magic("matplotlib", "widget")
    annotator = ManualVehicleAnnotator(image, GROUND_TRUTH_PATH)
    plt.show()
    return annotator


print("Ground truth editor is optional. Run annotator = launch_manual_annotation() to edit it.")

## Próximas células

Após concluir a revisão manual, este notebook carregará os detectores, executará os benchmarks controlados e apresentará as métricas e comparações visuais.

## Benchmarks reproduziveis

Execute as celulas abaixo somente apos revisar o ground truth. Cada configuracao usa a mesma imagem, o mesmo IoU de matching e tres repeticoes medidas depois de um warm-up.

In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd

from src.detection.sahi_detector import SahiVehicleDetector
from src.detection.yolo_detector import YoloVehicleDetector
from src.evaluation.benchmark import benchmark_detector
from src.evaluation.ground_truth import load_ground_truth
from src.roads.opencv_road import highlight_roads
from src.visualization.draw import (
    draw_detection_diagnostics,
    draw_detections,
    draw_ground_truth,
)

MODEL_DIR = ROOT / "models"
image_array = np.asarray(image)
ground_truth = load_ground_truth(GROUND_TRUTH_PATH)


def build_detector(
    model_name,
    domain,
    confidence=0.25,
    sahi_enabled=False,
    slice_size=512,
    image_size=1024,
    overlap=0.20,
):
    model_path = MODEL_DIR / model_name
    if not model_path.exists():
        raise FileNotFoundError(
            f"Model not found: {model_path}. Run scripts/download_models.py first."
        )
    if sahi_enabled:
        return SahiVehicleDetector(
            model_path,
            domain,
            confidence,
            image_size,
            slice_size,
            overlap,
            device="auto",
            merge_iou=0.50,
        )
    return YoloVehicleDetector(model_path, domain, confidence, image_size, 0.70, device="auto")


def run_benchmark_table(experiments):
    rows = []
    for name, detector_options in experiments:
        result = benchmark_detector(
            name, build_detector(**detector_options), image_array, ground_truth, repetitions=3
        )
        rows.append(result.as_dict())
    return pd.DataFrame(rows)


print(f"Ground truth reviewed: {len(ground_truth)} vehicles")

In [ ]:
size_experiments = [
    ("aerial_n_normal", {"model_name": "yolo11n-obb.pt", "domain": "aerial"}),
    ("aerial_s_normal", {"model_name": "yolo11s-obb.pt", "domain": "aerial"}),
    ("aerial_m_normal", {"model_name": "yolo11m-obb.pt", "domain": "aerial"}),
]
size_results = run_benchmark_table(size_experiments)
size_results.sort_values(["f1", "median_inference_ms"], ascending=[False, True])

In [ ]:
controlled_experiments = [
    ("coco_normal", {"model_name": "yolo11n.pt", "domain": "coco"}),
    ("coco_sahi_512", {"model_name": "yolo11n.pt", "domain": "coco", "sahi_enabled": True}),
    ("aerial_normal", {"model_name": "yolo11n-obb.pt", "domain": "aerial"}),
    ("aerial_sahi_512", {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True}),
]
controlled_results = run_benchmark_table(controlled_experiments)
controlled_results.sort_values("f1", ascending=False)

## Diagnostico de erros

Use esta visualizacao depois do benchmark 2x2 para revisar a configuracao candidata. Verde representa *true positive* (TP), amarelo *false positive* (FP) e vermelho *false negative* (FN), sempre usando o mesmo matching IoU >= 0.50 das metricas.

In [ ]:
diagnostic_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
diagnostic_detections = diagnostic_detector.predict(image_array)
diagnostic_image = draw_detection_diagnostics(
    image_array, diagnostic_detections, ground_truth, iou_threshold=0.50
)

figure, axis = plt.subplots(figsize=(14, 10))
axis.imshow(diagnostic_image)
axis.set_title("Detection audit: green=TP, amber=FP, red=FN")
axis.set_axis_off()
plt.tight_layout()
plt.show()
print("Green: TP | Amber: FP | Red: FN")

In [ ]:
threshold_experiments = [
    (
        f"aerial_sahi_conf_{confidence:.2f}",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "confidence": confidence,
            "sahi_enabled": True,
        },
    )
    for confidence in (0.15, 0.25, 0.35, 0.50)
]
threshold_results = run_benchmark_table(threshold_experiments)

slice_experiments = [
    (
        f"aerial_sahi_slice_{slice_size}",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "sahi_enabled": True,
            "slice_size": slice_size,
        },
    )
    for slice_size in (512, 640)
]
slice_results = run_benchmark_table(slice_experiments)
display(threshold_results.sort_values("f1", ascending=False))
display(slice_results.sort_values("f1", ascending=False))

In [ ]:
# One variable changes at a time relative to the 1024 px / 20% overlap control.
targeted_experiments = [
    (
        "aerial_sahi_control",
        {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True},
    ),
    (
        "aerial_sahi_imgsz_1280",
        {
            "model_name": "yolo11n-obb.pt",
            "domain": "aerial",
            "sahi_enabled": True,
            "image_size": 1280,
        },
    ),
    (
        "aerial_sahi_overlap_030",
        {"model_name": "yolo11n-obb.pt", "domain": "aerial", "sahi_enabled": True, "overlap": 0.30},
    ),
]
targeted_results = run_benchmark_table(targeted_experiments)
targeted_results.sort_values("f1", ascending=False)

In [ ]:
final_detector = build_detector("yolo11n-obb.pt", "aerial", confidence=0.15, sahi_enabled=True)
final_detections = final_detector.predict(image_array)

figure, axes = plt.subplots(1, 3, figsize=(22, 8))
for axis, title, rendered in zip(
    axes,
    ("Original", "Ground truth reviewed", "Final prediction"),
    (
        image_array,
        draw_ground_truth(image_array, ground_truth),
        draw_detections(image_array, final_detections),
    ),
    strict=True,
):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
print(f"Final predicted count: {len(final_detections)}")

## Laboratorio da malha viaria

Este experimento permanece somente no notebook. Ele compara o destaque HSV atual com uma hipotese classica: vias tendem a formar faixas longas horizontais ou verticais, portanto uma abertura morfologica retangular pode reduzir regioes pequenas. Nao e uma segmentacao semantica nem uma configuracao da aplicacao final.

In [ ]:
import cv2


def overlay_mask(image, mask, color=(30, 197, 226), opacity=0.35):
    rendered = image.copy()
    selected = mask > 0
    rendered[selected] = (image[selected] * (1 - opacity) + np.asarray(color) * opacity).astype(
        np.uint8
    )
    return rendered


def linear_road_candidate(image):
    # Same HSV candidate as the app; only the geometric post-processing changes.
    hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
    candidate = cv2.inRange(
        hsv, np.asarray([5, 35, 45], dtype=np.uint8), np.asarray([35, 95, 190], dtype=np.uint8)
    )
    base_kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    cleaned = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, base_kernel)
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_OPEN, base_kernel)

    horizontal = cv2.morphologyEx(
        cleaned, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (81, 9))
    )
    vertical = cv2.morphologyEx(
        cleaned, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (9, 81))
    )
    linear = cv2.bitwise_or(horizontal, vertical)
    return cv2.morphologyEx(
        linear,
        cv2.MORPH_CLOSE,
        cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)),
    )


baseline_roads = highlight_roads(
    image_array, hsv_lower=(5, 35, 45), hsv_upper=(35, 95, 190), kernel_size=7, min_area=1500
)
linear_mask = linear_road_candidate(image_array)

figure, axes = plt.subplots(2, 2, figsize=(18, 14))
views = [
    ("Original", image_array),
    ("Vehicle detections (final configuration)", draw_detections(image_array, final_detections)),
    ("Road baseline: HSV + morphology", baseline_roads.overlay),
    ("Experiment: HSV + linear continuity", overlay_mask(image_array, linear_mask)),
]
for axis, (title, rendered) in zip(axes.flat, views, strict=True):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
plt.show()

for name, mask in [("baseline", baseline_roads.mask), ("linear experiment", linear_mask)]:
    coverage = 100 * np.count_nonzero(mask) / mask.size
    print(f"{name}: {coverage:.2f}% of image highlighted")

## Experimento neural: segmentacao semantica de vias

A mascara HSV anterior confunde pavimento com telhados e solo exposto porque usa somente cor. Nesta etapa testamos, **somente no notebook**, o checkpoint [`mfaytin/mask2former-satellite`](https://huggingface.co/mfaytin/mask2former-satellite), um Mask2Former treinado no OpenEarthMap para classes de cobertura do solo. O objetivo e verificar se contexto visual (rua, vegetacao, construcao e solo) reduz esses falsos positivos.

O modelo nao foi treinado nesta imagem e nenhum fine-tuning sera feito agora. Ha diferenca entre a resolucao/origem do treino e a foto de drone; portanto a comparacao e qualitativa e nao autoriza, por si so, alterar o pipeline final.

### 1. Carregar checkpoint e imagem

Esta celula usa CUDA quando disponivel, baixa os pesos apenas na primeira execucao e armazena o cache em `cache/huggingface/` (ignorado pelo Git). O checkpoint possui IDs genericos (`LABEL_n`) no arquivo de configuracao; a proxima celula mostra todas as classes preditas para validar visualmente qual ID corresponde a via nesta imagem.

In [ ]:
from time import perf_counter

import torch
from PIL import Image
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

SEGMENTATION_MODEL_ID = "mfaytin/mask2former-satellite"
SEGMENTATION_ROAD_CLASS_ID = 3
SEGMENTATION_CACHE_DIR = ROOT / "cache" / "huggingface"
SEGMENTATION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
SEGMENTATION_DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


def neural_overlay(image, mask, color=(30, 197, 226), opacity=0.40):
    rendered = image.copy()
    selected = mask > 0
    rendered[selected] = (image[selected] * (1 - opacity) + np.asarray(color) * opacity).astype(
        np.uint8
    )
    return rendered


segmentation_processor = Mask2FormerImageProcessor.from_pretrained(
    SEGMENTATION_MODEL_ID, cache_dir=SEGMENTATION_CACHE_DIR
)
segmentation_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    SEGMENTATION_MODEL_ID, cache_dir=SEGMENTATION_CACHE_DIR
).to(SEGMENTATION_DEVICE)
segmentation_model.eval()
segmentation_input = Image.fromarray(image_array)

print(f"Segmentation device: {SEGMENTATION_DEVICE}")
print(f"Model: {SEGMENTATION_MODEL_ID}")
print("Run the class audit below before interpreting the road mask.")

### 2. Inferir e auditar os IDs de classe

O mapa e reamostrado para as dimensoes originais da imagem. Como o checkpoint nao fornece nomes confiaveis para os IDs, esta celula destaca cada classe predita com uma cor diferente. Nesta cena, a auditoria visual identificou `LABEL_3` como a via: ela acompanha a rua central e os acessos, enquanto `LABEL_4` acompanha arvores. Mantenha essa celula como evidencia da decisao; nao presuma que o mesmo ID sera valido para checkpoints diferentes.

In [ ]:
segmentation_inputs = segmentation_processor(images=segmentation_input, return_tensors="pt")
segmentation_inputs = {
    name: value.to(SEGMENTATION_DEVICE) for name, value in segmentation_inputs.items()
}
started_at = perf_counter()
with torch.inference_mode():
    segmentation_outputs = segmentation_model(**segmentation_inputs)
segmentation_ms = (perf_counter() - started_at) * 1_000
segmentation_map = (
    segmentation_processor.post_process_semantic_segmentation(
        segmentation_outputs, target_sizes=[image_array.shape[:2]]
    )[0]
    .detach()
    .cpu()
    .numpy()
)

audit_colors = [(0, 140, 255), (255, 193, 7), (0, 200, 83), (244, 67, 54), (156, 39, 176)]
predicted_class_ids = np.unique(segmentation_map)
figure, axes = plt.subplots(1, len(predicted_class_ids), figsize=(5 * len(predicted_class_ids), 5))
for index, (axis, class_id) in enumerate(zip(axes, predicted_class_ids, strict=True)):
    class_mask = segmentation_map == class_id
    rendered = neural_overlay(
        image_array,
        class_mask.astype(np.uint8) * 255,
        color=audit_colors[index % len(audit_colors)],
        opacity=0.45,
    )
    axis.imshow(rendered)
    axis.set_title(f"LABEL_{class_id}: {100 * class_mask.mean():.1f}%")
    axis.set_axis_off()
plt.tight_layout()
plt.show()
print(f"Segmentation inference: {segmentation_ms:.1f} ms")
print(f"Predicted class IDs: {predicted_class_ids.tolist()}")

### 3. Comparar veiculos, HSV e segmentacao

Execute antes a celula `final-visual-comparison`, que produz `final_detections`. Esta comparacao coloca a previsao final de veiculos ao lado da camada HSV atual e da mascara neural. Revise principalmente: (1) a rua principal e o cruzamento devem estar destacados; (2) telhados, areia e barro nao devem dominar a mascara; e (3) o resultado ainda pode conter pequenos falsos positivos e trechos ausentes.

In [ ]:
neural_road_mask = (segmentation_map == SEGMENTATION_ROAD_CLASS_ID).astype(np.uint8) * 255
neural_road_overlay = neural_overlay(image_array, neural_road_mask, opacity=0.40)
neural_road_coverage = 100 * np.count_nonzero(neural_road_mask) / neural_road_mask.size
neural_baseline_roads = highlight_roads(
    image_array, hsv_lower=(5, 35, 45), hsv_upper=(35, 95, 190), kernel_size=7, min_area=1500
)
baseline_road_coverage = (
    100 * np.count_nonzero(neural_baseline_roads.mask) / neural_baseline_roads.mask.size
)

figure, axes = plt.subplots(1, 4, figsize=(26, 7))
views = [
    ("Original", image_array),
    ("Vehicle detections (final configuration)", draw_detections(image_array, final_detections)),
    ("Road baseline: HSV + morphology", neural_baseline_roads.overlay),
    ("Road experiment: Mask2Former LABEL_3", neural_road_overlay),
]
for axis, (title, rendered) in zip(axes, views, strict=True):
    axis.imshow(rendered)
    axis.set_title(title)
    axis.set_axis_off()
plt.tight_layout()
plt.show()
print(f"Mask2Former road coverage: {neural_road_coverage:.2f}% of image")
print(f"HSV baseline coverage: {baseline_road_coverage:.2f}% of image")
print("Qualitative result only: no road ground truth or IoU is available yet.")

### Leitura preliminar desta imagem

No ambiente local, a mascara neural `LABEL_3` cobriu 5,59% da imagem, contra 34,91% da mascara HSV atual. Na revisao visual, ela recuperou a rua principal, o cruzamento e acessos relevantes sem classificar telhados e grandes areas de barro como via. Ainda existem pequenos falsos positivos e possiveis trechos ausentes.

Esse e um resultado qualitativo de uma unica imagem. Antes de integrar o modelo ao pipeline, a proxima evidencia necessaria e um ground truth de via em imagens independentes para calcular IoU, precision e recall de segmentacao.

## Registro da decisao

Os resultados executados nesta imagem estao registrados em `outputs/metrics/benchmark_results.json`. A configuracao final so deve ser atualizada depois de comparar F1, recall, erro absoluto de contagem e tempo. Como o ground truth foi iniciado por pre-anotacoes do modelo aereo e revisado manualmente, os resultados sao evidencias para esta prova, nao uma estimativa de generalizacao para novas imagens.